In [7]:
# ============================================================
# PACKAGE 12 : CITIZEN HEALTH ADVISORY AGENT
# VERSION 3.0
# PART 1/3
#
# Purpose:
# - Load Package 10.2 Ward Source Attribution output
# - Load sensitive locations
# - Load AQI forecast output
# - Create ward-level health risk dataframe
# - Save Ward_Health_Risk.csv
#
# ============================================================


import os
import json
import pandas as pd
import geopandas as gpd
import numpy as np


print("="*75)
print("PACKAGE 12 VERSION 3.0")
print("CITIZEN HEALTH ADVISORY AGENT")
print("PART 1/3")
print("="*75)



# ============================================================
# CONFIGURATION
# ============================================================



WARD_ATTRIBUTION_FILE = (r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Geospatial_Pollution_Source_Attribution_Engine\Ward_Level_Source_Attribution_Outputs\Ward_Source_Attribution.csv")

SENSITIVE_FILE = (r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Cleaning\cleaned_sensitive_locations.geojson")


FORECAST_FILE = (r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Last_Stage\Submission_Outputs\Frontend_Model_Output.json")



OUTPUT_DIR = (r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Citizen_Health_Advisory\Part1")


os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)



OUTPUT_WARD_HEALTH = os.path.join(
    OUTPUT_DIR,
    "Ward_Health_Risk.csv"
)



# ============================================================
# LOAD DATA
# ============================================================


print("\nLoading Data...")


# ----------------------------
# Ward Attribution
# ----------------------------

if not os.path.exists(WARD_ATTRIBUTION_FILE):

    raise FileNotFoundError(
        "Ward_Source_Attribution.csv not found"
    )


ward_df = pd.read_csv(
    WARD_ATTRIBUTION_FILE
)



print("\nWard Attribution:")
print(
    ward_df.shape
)



# ----------------------------
# Sensitive Locations
# ----------------------------

if not os.path.exists(SENSITIVE_FILE):

    raise FileNotFoundError(
        "cleaned_sensitive_locations.geojson not found"
    )


sensitive_gdf = gpd.read_file(
    SENSITIVE_FILE
)



print("\nSensitive Locations:")
print(
    sensitive_gdf.shape
)



# ----------------------------
# Forecast Output
# ----------------------------


if not os.path.exists(FORECAST_FILE):

    raise FileNotFoundError(
        "Frontend_Model_Output.json not found"
    )



with open(
    FORECAST_FILE,
    "r"
) as f:

    forecast_json = json.load(f)



print("\nForecast File Loaded")



# ============================================================
# EXTRACT AQI VALUE SAFELY
# ============================================================


forecast_aqi = None



possible_keys = [

    "average_predicted_AQI",
    "Average_Predicted_AQI",
    "predicted_AQI",
    "AQI",
    "forecast_AQI"

]



for key in possible_keys:

    if key in forecast_json:

        forecast_aqi = forecast_json[key]

        break



if forecast_aqi is None:


    # fallback:
    # if prediction list exists

    if "predictions" in forecast_json:

        forecast_aqi = np.mean(
            forecast_json["predictions"]
        )


    elif "Predictions" in forecast_json:

        forecast_aqi = np.mean(
            forecast_json["Predictions"]
        )



if forecast_aqi is None:

    raise Exception(
        "AQI value not found in Frontend_Model_Output.json"
    )



print(
    "\nForecast AQI:",
    forecast_aqi
)



# ============================================================
# AQI HEALTH RISK CATEGORY
# ============================================================


def aqi_risk_category(aqi):


    if aqi <= 50:

        return "Good"


    elif aqi <= 100:

        return "Satisfactory"


    elif aqi <= 200:

        return "Moderate"


    elif aqi <= 300:

        return "Poor"


    elif aqi <= 400:

        return "Very Poor"


    else:

        return "Severe"



global_aqi_category = aqi_risk_category(
    forecast_aqi
)



print(
    "AQI Category:",
    global_aqi_category
)



# ============================================================
# PREPARE WARD DATA
# ============================================================


health_df = ward_df.copy()



# Normalize column names

health_df.columns = [

    c.strip()

    for c in health_df.columns

]



# ============================================================
# ENSURE REQUIRED COLUMNS EXIST
# ============================================================


required_columns = [

    "Traffic_%",
    "Construction_%",
    "Industrial_%",
    "Sensitive_Count",
    "Dominant_Source",
    "Confidence"

]



for col in required_columns:


    if col not in health_df.columns:


        health_df[col] = 0



# Convert numeric columns


numeric_columns = [

    "Traffic_%",
    "Construction_%",
    "Industrial_%",
    "Sensitive_Count"

]



for col in numeric_columns:


    health_df[col] = pd.to_numeric(

        health_df[col],

        errors="coerce"

    ).fillna(0)



print(
    "\nHealth dataframe prepared"
)


print(
    health_df.shape
)



# ============================================================
# SENSITIVE LOCATION NORMALIZATION
# ============================================================


total_sensitive_locations = len(
    sensitive_gdf
)



print(
    "\nTotal Sensitive Locations:",
    total_sensitive_locations
)



if health_df["Sensitive_Count"].max() > 0:

    health_df["Sensitivity_Score"] = (

        health_df["Sensitive_Count"]

        /

        health_df["Sensitive_Count"].max()

    )


else:

    health_df["Sensitivity_Score"] = 0



# ============================================================
# POLLUTION SEVERITY SCORE
# ============================================================


aqi_factor = min(

    forecast_aqi / 500,

    1

)



health_df["AQI_Severity_Score"] = (

    aqi_factor * 100

)



# ============================================================
# SOURCE IMPACT SCORE
# ============================================================


health_df["Pollution_Source_Score"] = (

    health_df["Traffic_%"] * 0.4

    +

    health_df["Construction_%"] * 0.3

    +

    health_df["Industrial_%"] * 0.3

)



# ============================================================
# HEALTH RISK SCORE
# ============================================================


health_df["Health_Risk_Score"] = (

    health_df["AQI_Severity_Score"] * 0.5

    +

    health_df["Pollution_Source_Score"] * 0.3

    +

    health_df["Sensitivity_Score"] * 100 * 0.2

)



# Limit range

health_df["Health_Risk_Score"] = (

    health_df["Health_Risk_Score"]

    .clip(0,100)

)



# ============================================================
# HEALTH RISK LEVEL
# ============================================================


def health_level(score):


    if score < 25:

        return "Low"


    elif score < 50:

        return "Moderate"


    elif score < 75:

        return "High"


    else:

        return "Critical"



health_df["Health_Risk_Level"] = (

    health_df["Health_Risk_Score"]

    .apply(
        health_level
    )

)



# ============================================================
# SAVE INTERMEDIATE OUTPUT
# ============================================================


health_df.to_csv(

    OUTPUT_WARD_HEALTH,

    index=False

)



print(
    "\nSaved:"
)

print(
    OUTPUT_WARD_HEALTH
)



print(
    "\nPART 1 COMPLETED SUCCESSFULLY"
)

PACKAGE 12 VERSION 3.0
CITIZEN HEALTH ADVISORY AGENT
PART 1/3

Loading Data...

Ward Attribution:
(290, 15)

Sensitive Locations:
(983, 5)

Forecast File Loaded

Forecast AQI: 181.68923950195312
AQI Category: Moderate

Health dataframe prepared
(290, 15)

Total Sensitive Locations: 983

Saved:
C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Citizen_Health_Advisory\Part1\Ward_Health_Risk.csv

PART 1 COMPLETED SUCCESSFULLY
